# 10.3 排查路径与决策树

上一节是「地图」，本节是「导航」：拿到一个性能问题，**第一步做什么、判断什么、走哪条分支**——全部固化成决策树与对照表。

## 1. 总决策树

```
拿到问题（实测 vs 理论差距）
│
├─ 第 0 步: 复测确认
│    巏别是波动还是稳定可复现；锁 shape、锁频率、取多次中位数
│
├─ 第 1 步: 建模定位（7.2）
│    算 I = W/Q，画 Roofline
│    ├─ 带宽受限（I < I*）→ 走「访存线」
│    └─ 计算受限（I > I*）→ 走「计算线」
│
├─ 第 2 步: 核间均衡检查（9.2~9.5）
│    看 per-core 耗时分布（mskpp 采集）
│    ├─ 分布不均 → 负载均衡问题 → 分核策略复查
│    └─ 分布均匀但都慢 → 往下
│
├─ 第 3 步: 引擎间配平（7.5）
│    看 Cube/Vector 各自 busy/idle 比例
│    ├─ 一方长期 idle → CV 流水问题 → 四拍/preload/块大小
│    └─ 两方都 busy → 往下
│
├─ 第 4 步: 引擎内逐项（8.2~8.6）
│    [带宽线] Cube K 切分 / Vector DN / bank 冲突（7.3）
│    [计算线] VF 五式 / vLoop / RegBase（7.6）
│    通用: Scalar 上提/增量化/分支消除（8.6）
│
└─ 第 5 步: 验证回滚
     每改一项就复测对比；没收益就回滚, 别攒大提交
```

**两条线的区别**：带宽受限时优化计算指令是白干；计算受限时抠搬运也是白干——第 1 步的判断决定后面所有分支的优先级。

## 2. 症状 → 处方对照表

| 症状（工具可见的现象） | 最可能的原因 | 处方 | 出处 |
|--|--|--|--|
| 各核耗时方差大 | 分核不均（mask/section 没计入） | 有效块计量 + 贪心分核 | [9.2/9.3](../09_performance_high_level/09.02_imbalance_sources.ipynb) |
| decode 时大量核空转 | 并行度不足 | 切 S2 + merge | [9.5](../09_performance_high_level/09.05_s2_merge.ipynb) |
| Vector busy 低、Cube busy 高 | Vector 太快/太少活 | 块大小重权衡、AIV 配比 | [7.5](../07_performance_basics/07.05_cv_pipeline.ipynb) |
| Cube 断流（idle 尖刺） | Vector 长阶段没掩盖 | preload 流水 | [5章](../05_flash_attn_non_quantized/README.md) |
| MTE1 实际带宽远低于标称 | bank 冲突 / 整块率低 | 三板斧；K 切分复查 | [7.3](../07_performance_basics/07.03_memory_performance.ipynb) / [8.2](../08_performance_advanced/08.02_cube_k_split.ipynb) |
| dump log 有 BGBK_CFLT | UB bank 冲突实锤 | 指令限宽/pingpong/多 bank 组 | [7.3](../07_performance_basics/07.03_memory_performance.ipynb) |
| L0 装载失败/降级 | 容量约束没算对 | K 切分容量方程重算 | [8.3](../08_performance_advanced/08.03_cube_best_k.ipynb) |
| VF 段耗时与建模差大 | bank 冲突或依赖等待未编排 | 双缓冲 / 编排复查 | [8.5](../08_performance_advanced/08.05_vector_dn_modeling.ipynb) |
| 分支密集处流水抖 | Scalar 分支 | 掩码/查表/模板特化 | [8.6](../08_performance_advanced/08.06_scalar_opt.ipynb) |

## 3. 贯穿案例完整走查

回到「decode 3.2ms vs 理论 1.1ms」：

In [ ]:
# 动手: 贯穿案例 —— 按决策树走一遍排查
# 模拟一个 decode 场景的画像数据与排查判断

profile = {
    'shape':         'B=1, N=8, S1=1, S2=8192 (decode)',
    'roofline_I':    12,          # 计算强度: 每字节 FLOPs
    'knee_I':        85,           # 拐点 I* = Pm/Bm
    'core_dist':     [3.2] + [0.9]*7,   # 各核耗时(ms): 核0 3.2, 其余 0.9
    'cube_busy':     0.30,
    'vector_busy':  0.28,
}

# ---- 第 1 步: 建模定位 ----
bound = '带宽受限' if profile['roofline_I'] < profile['knee_I'] else '计算受限'
print(f"第1步 建模: I={profile['roofline_I']} < I*={profile['knee_I']} -> {bound}")

# ---- 第 2 步: 核间均衡 ----
d = profile['core_dist']
spread = max(d) / min(d)
print(f"第2步 均衡: 核耗时极差 {spread:.1f}x -> 严重不均, 核间问题!")
print(f"          decode 并行度 = B*N*S1 = 8 < 核数 8... 但核0 独扛 3.2ms")
print(f"          处方: 切 S2 + merge (flash decoding) —— 见 9.5")
print()

# ---- 切 S2 之后推演 ----
print("推演: 切 S2 后 8 核各分 1/8 的 KV 段, 核间负载拉平")
est = max(d) / 8 + 0.15   # merge 开销 ~0.15ms
print(f"      预计耗时 ≈ 3.2/8 + 0.15(merge) = {est:.2f} ms")
print(f"      若仍 > 理论 1.1ms, 继续第3/4步: 查 CV 配平与 DN 通路")


### 观察结论

决策树的每一步都**把「感觉」换成「证据」**：建模给 bound 判断、核间给耗时分布、引擎间给 busy 比例、指令级给 dump log——四层证据一层层收敛，最后落在一张具体的处方上。

## 4. 排查的四条纪律

1. **先测量再动手**：没有画像不改代码（[7.1](../07_performance_basics/07.01_chapter_intro.ipynb) 的第一原则）；
2. **一次只改一项**：改两项就说不清是谁的收益；
3. **每项都有量化验收**：预期收益写下来，复测对不上就回滚；
4. **回到上一层**：下层优化吃满后，重新测整机画像——瓶颈位置会迁移。

第 4 条最容易被忽略：切完 S2 之后瓶颈可能从核间搬到引擎间甚至 bank 冲突上，**优化是迭代过程，不是一次性任务**。

## 本节要点

- 总决策树：复测确认 → 建模定位（带宽/计算线）→ 核间均衡 → 引擎间配平 → 引擎内逐项 → 验证回滚；
- 症状对照表把 dump log 的可见现象直接映射到处方；
- 四条纪律：先测量、单项修改、量化验收、瓶颈迁移后回到上一层重查。

## 小测验

1. 决策树第 1 步为什么必须先判 bound？两条线分别优先查什么？
2. 「各核耗时方差大」和「Cube busy 低但 Vector busy 高」分别对应什么处方？
3. 排查四条纪律是什么？为什么「回到上一层」最重要？
4. 优化为什么是迭代过程？瓶颈迁移是什么意思？

>>> 参考答案见 [answer/10.03_answer.txt](answer/10.03_answer.txt)
